In [ ]:
import pandas as pd
from collections import defaultdict

# 读取Excel文件
df = pd.read_excel('师生双选信息表.xlsx', header=0)
data = df.values.tolist()

# 创建反向映射字典
teacher_group_map = {}
groups = {
    "1组": ["马晶军", "李现", "纪晓婧", "芦爽", "张斌"],
    "2组": ["周欣", "刘红蕾", "高贵", "闫思琦", "刘莎莎"],
    "3组": ["许国贺", "付美臻", "彭丹丹"],
    "4组": ["刘祺凤", "孙燕芳", "代瑞慧"],
    "5组": ["王林艳", "郭利静", "陈明珠"],
    "6组": ["张力红", "李继文", "石田"],
    "7组": ["周晓斐", "高建平", "赵丽丽"],
}
for group, teachers in groups.items():
    for teacher in teachers:
        teacher_group_map[teacher] = group

# 创建学生列表（包含完整信息）
columns = ["学生", "学号", "学生所在院系", "学生所在专业", "学生所在班级", "题目", "指导教师"]
students = []
for row in data:
    students.append({"学生": row[0], "学号": str(row[1]).strip(), "学生所在院系": row[2], "学生所在专业": row[3], "学生所在班级": row[4], "题目": row[5], "指导教师": row[6]})

# 初始化分配结果
group_students = {g: [] for g in groups}

# 分配算法
teacher_students = defaultdict(list)
for s in students:
    teacher_students[s["指导教师"]].append(s)

for teacher, t_students in teacher_students.items():
    forbidden_group = teacher_group_map.get(teacher, "")
    available_groups = [g for g in groups if g != forbidden_group]

    for student in t_students:
        candidate_groups = sorted(available_groups, key=lambda x: len(group_students[x]))
        selected_group = candidate_groups[0]
        group_students[selected_group].append(student)

# 平衡调整
while True:
    group_sizes = [(g, len(s)) for g, s in group_students.items()]
    max_group = max(group_sizes, key=lambda x: x[1])
    min_group = min(group_sizes, key=lambda x: x[1])

    if max_group[1] - min_group[1] <= 1:
        break

    for student in group_students[max_group[0]]:
        if teacher_group_map[student["指导教师"]] != min_group[0]:
            group_students[max_group[0]].remove(student)
            group_students[min_group[0]].append(student)
            break

# 创建合并数据表
final_data = []
for group_name, members in group_students.items():
    for student in members:
        student_data = {k: v for k, v in student.items()}
        student_data["答辩组"] = group_name
        final_data.append(student_data)

result_df = pd.DataFrame(final_data)
column_order = ["答辩组"] + columns
result_df = result_df[column_order]

# 创建排序后的数据表（按答辩组、专业、班级排序）
sorted_df = result_df.sort_values(by=["答辩组", "学生所在专业", "学生所在班级"])

# 写入Excel文件
with pd.ExcelWriter("答辩分组.xlsx") as writer:
    # Sheet1：原始分配结果
    result_df.to_excel(writer, sheet_name="原始分组", index=False)
    # 设置列宽
    worksheet = writer.sheets["原始分组"]
    for idx, col in enumerate(result_df.columns):
        max_len = max(result_df[col].astype(str).apply(len).max(), len(col)) + 2
        worksheet.column_dimensions[chr(65 + idx)].width = max_len

    # Sheet2：排序后的结果
    sorted_df.to_excel(writer, sheet_name="排序分组", index=False)
    worksheet = writer.sheets["排序分组"]
    for idx, col in enumerate(sorted_df.columns):
        max_len = max(sorted_df[col].astype(str).apply(len).max(), len(col)) + 2
        worksheet.column_dimensions[chr(65 + idx)].width = max_len

    # 添加分组统计
    stats = result_df.groupby("答辩组").size().reset_index(name="人数")
    stats.to_excel(writer, sheet_name="分组统计", index=False)

print("文件已生成：答辩分组.xlsx")
print("包含两个工作表：")
print("1. 原始分组 - 原始分配顺序")
print("2. 排序分组 - 按答辩组、专业、班级排序")
print(f"总学生数：{len(result_df)}")
print(f"分组情况：\n{stats}")

文件已生成：答辩分组.xlsx
包含两个工作表：
1. 原始分组 - 原始分配顺序
2. 排序分组 - 按答辩组、专业、班级排序
总学生数：285
分组情况：
  答辩组  人数
0  1组  41
1  2组  41
2  3组  40
3  4组  41
4  5组  41
5  6组  41
6  7组  40


In [6]:
import pandas as pd
from collections import defaultdict

# 读取Excel文件
df = pd.read_excel('师生双选信息表.xlsx', header=0)
df

,学生,学号,学生所在院系,学生所在专业,学生所在班级,题目,指导教师
0,鲍元昊,2021984150402,理工系,化学工程与工艺,2102班,年产5万吨二乙酸亚乙酯生产工艺设计,刘祺凤
1,段可欣,2021984150121,理工系,化学工程与工艺,2103班,12万吨/年乙烯法制乙酸乙烯酯生产工艺设计,刘祺凤
2,耿雨森,2021984150408,理工系,化学工程与工艺,2104班,年产2万吨二甲基亚砜生产工艺设计,刘祺凤
3,郭康辉,2021984151003,理工系,化学工程与工艺,2101班,年产6.5万吨丙烯氧化法制丙烯醛生产工艺设计,刘祺凤
4,刘英英,2021984150126,理工系,化学工程与工艺,2104班,5万吨/年乙烯氧氯化制环氧乙烷生产工艺设计,刘祺凤
...,...,...,...,...,...,...,...
280,白松康,2021984150401,理工系,化学工程与工艺,2101班,年产8万吨乙醛肟工艺设计,马晶军
281,李松浩,2021984150608,理工系,制药工程,2104班,年产1000万袋0.9%氯化钠注射剂（软袋）的生产车间工艺设计,马晶军
282,李天琪,2021984150710,理工系,制药工程,2102班,年产1.6亿瓶10%葡萄糖注射剂制剂工段车间工艺设计,马晶军
283,张少龙,2019984060403,理工系,化学工程与工艺,2110班,年产10000吨丁酮肟生产工艺设计,马晶军
